In [0]:

# ============================================================
# CELDA 1. EDA GOLD: DISTRIBUCIÓN Y EVOLUCIÓN DE RETRASOS
# ============================================================

# Analizamos los retrasos de salida de vuelos mediante:
# 1. Distribución por categorías de retraso.
# 2. Evolución diaria del retraso promedio.
# 3. Variabilidad diaria de los retrasos.
#
# Utilizamos las tablas Gold que preparamos anteriormente.
# No necesitamos volver a procesar los vuelos originales.

from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CARGAR LAS TABLAS GOLD
# ------------------------------------------------------------

distribucion = spark.table(
    "workspace.gold_flights.delay_distribution"
)

serie_diaria = spark.table(
    "workspace.gold_flights.delay_daily_timeseries"
)

# Estas tablas ya están agregadas, así que podemos
# convertirlas a pandas para crear las gráficas.

df_dist = distribucion.toPandas()
df_diaria = serie_diaria.orderBy("flight_date").toPandas()

# ------------------------------------------------------------
# 2. DISTRIBUCIÓN POR CATEGORÍAS DE RETRASO
# ------------------------------------------------------------

orden_categorias = [
    "Salida anticipada",
    "0 a 14 minutos",
    "15 a 29 minutos",
    "30 a 59 minutos",
    "60 a 119 minutos",
    "120 minutos o más"
]

df_dist["delay_category"] = __import__("pandas").Categorical(
    df_dist["delay_category"],
    categories=orden_categorias,
    ordered=True
)

df_dist = df_dist.sort_values("delay_category")

plt.figure(figsize=(11, 5))

plt.bar(
    df_dist["delay_category"].astype(str),
    df_dist["percentage"]
)

plt.title("Distribución de los retrasos de salida")
plt.xlabel("Categoría de retraso")
plt.ylabel("Porcentaje de vuelos (%)")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. EVOLUCIÓN DEL RETRASO PROMEDIO DIARIO
# ------------------------------------------------------------

# Suavizamos la visualización con una media móvil de
# siete días para observar mejor la tendencia.
# Esta media móvil se utiliza solo para describir
# los datos históricos, no como variable predictora.

df_diaria["flight_date"] = __import__("pandas").to_datetime(
    df_diaria["flight_date"]
)

df_diaria["moving_avg_7d"] = (
    df_diaria["avg_delay_minutes"]
    .rolling(window=7, min_periods=1)
    .mean()
)

plt.figure(figsize=(13, 5))

plt.plot(
    df_diaria["flight_date"],
    df_diaria["avg_delay_minutes"],
    alpha=0.3,
    label="Retraso promedio diario"
)

plt.plot(
    df_diaria["flight_date"],
    df_diaria["moving_avg_7d"],
    linewidth=2,
    label="Media móvil de 7 días"
)

plt.title("Evolución diaria de los retrasos")
plt.xlabel("Fecha")
plt.ylabel("Retraso promedio (minutos)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. VARIABILIDAD DIARIA DE LOS RETRASOS
# ------------------------------------------------------------

# La desviación estándar permite observar qué días
# presentan una mayor dispersión en los retrasos.
#
# Una desviación alta puede significar que los retrasos
# fueron muy diferentes entre los vuelos de ese día.

plt.figure(figsize=(13, 5))

plt.plot(
    df_diaria["flight_date"],
    df_diaria["std_delay_minutes"],
    linewidth=1
)

plt.title("Variabilidad diaria de los retrasos")
plt.xlabel("Fecha")
plt.ylabel("Desviación estándar (minutos)")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. INDICADORES PRINCIPALES
# ------------------------------------------------------------

print("=" * 55)
print("EDA GOLD: DISTRIBUCIÓN Y SERIES TEMPORALES")
print("=" * 55)

print("\nDISTRIBUCIÓN DE RETRASOS")
display(
    df_dist[
        [
            "delay_category",
            "total_flights",
            "percentage"
        ]
    ]
)

print("\nDÍAS CON MAYOR RETRASO PROMEDIO")
display(
    df_diaria[
        [
            "flight_date",
            "total_flights",
            "avg_delay_minutes",
            "std_delay_minutes",
            "delay_rate"
        ]
    ]
    .sort_values(
        "avg_delay_minutes",
        ascending=False
    )
    .head(10)
)

print("\nDÍAS CON MAYOR VARIABILIDAD")
display(
    df_diaria[
        [
            "flight_date",
            "total_flights",
            "std_delay_minutes"
        ]
    ]
    .sort_values(
        "std_delay_minutes",
        ascending=False
    )
    .head(10)
)

print("\nEDA GOLD DE SCRAPPY COMPLETADO.")
